# 03 - KnottedGraph vs Topoly scaling

This notebook compares KnottedGraph and Topoly on published reference embeddings. It is a benchmark notebook, not an introductory tutorial: run **Yamada Sanity Checks** first if you want a smaller correctness check.

The benchmark uses paper-reference geometry as the computational input:

- **Dobrynin--Vesnin theta graphs:** the canonical diagram from the two-strand torus diagram plus the exterior arc.
- **Li--Lei--Li--Vesnin edge-replacement families:** the explicit `infinity_+` replacement families from the published examples.

For each case the workflow is:

$$
\text{reference embedding}
\to \text{PD code}
\to \text{published polynomial check}
\to \text{KnottedGraph timing}
\to \text{Topoly timing or recorded skip}.
$$

The CSV under `User_guide/benchmarks/results/` is the resume ledger. KnottedGraph is recomputed for every requested case; cached Topoly measurements are reused by `case_key`, and later Topoly calls in a family are skipped after the first non-pass. Rows are replaced by `case_key`, so resumed runs do not accumulate duplicates.

**How to read the result:** KnottedGraph must match the published polynomial before a timing row is treated as a pass. Topoly rows document the behavior of the tested Topoly version on the same PD data and should be read together with the note below.


## 1. Load dependencies and locate the repository

This setup cell imports the benchmark machinery, finds the repository root, and prepares the result/cache paths used by the rest of the notebook.


In [ ]:
from pathlib import Path
import contextlib
import csv
import hashlib
import io
import json
import os
import sys
import time

from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook from inside the repository checkout."
    )

DEV = ROOT / "dev"
if str(DEV) not in sys.path:
    sys.path.insert(0, str(DEV))

RESULT_SCHEMA_VERSION = 9
RESULTS_DIR = ROOT / "User_guide" / "benchmarks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "03_knottedgraph_vs_topoly_scaling_rows.csv"

# Persistent geometric-projection cache. This caches only the deterministic
# 3D-embedding -> PDCode preprocessing. It NEVER caches the KnottedGraph Yamada
# evaluation itself.
PROJECTION_CACHE_SCHEMA = 1
PROJECTION_CACHE_DIR = RESULTS_DIR / "03_pdcode_cache"
PROJECTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REUSE_PROJECTION_CACHE = True
WRITE_PROJECTION_CACHE = True

# Old code used 120 samples for every local infinity_+ replacement. 24 is an
# even number (so t=1/2 is not a sampled vertex) and is much cheaper for PDCode.
# Correctness is still guarded by exact projected-crossing and theorem checks.
LLLV_INFINITY_SAMPLES = 24

RESUME_FROM_CSV = True
# Resume policy: KnottedGraph is ALWAYS recomputed.  Existing Topoly results
# are reused case-by-case and are never recomputed when already present.
REUSE_TOPOLY_FROM_CSV = True
# Once Topoly has an FAIL or ERROR for one size in a family, do not invoke
# Topoly at larger sizes in that family.  Those rows are still written with
# Topoly = ERROR while KnottedGraph continues normally.
STOP_TOPOLY_AFTER_FAMILY_NONPASS = True
RESET_CURRENT_RESULTS = False
SHOW_INTERNAL_JSON_LOGS = False
LIVE_STAGE_LOGS = True

PREFERRED_COLUMNS = [
    "settings_hash",
    "schema_version",
    "benchmark",
    "suite",
    "case_key",
    "graph_type",
    "size",
    "label",
    "family",
    "n",
    "abstract_graph",
    "V",
    "E",
    "crossings",
    "extra_projection_crossings",
    "knottedgraph_result",
    "knottedgraph_status",
    "knottedgraph_s",
    "topoly_result",
    "topoly_status",
    "topoly_s",
    "topoly_over_knottedgraph",
    "projection_source",
    "projection_s",
    "projection_cache_key",
    "case_wall_s",
    "timeout_s",
    "knottedgraph_timeout_s",
    "topoly_timeout_s",
    "row_error",
    "knottedgraph_error",
    "topoly_error",
    "paper",
    "theorem",
    "formula",
    "embedding_variant",
    "embedding_source",
    "reference_figure",
    "replacement",
    "max_degree",
    "pieces",
    "case_hash",
    "embedding_hash",
    "pd_hash",
]


def _settings_hash(payload):
    encoded = json.dumps(payload, sort_keys=True, default=str).encode()
    return hashlib.sha256(encoded).hexdigest()[:12]


def _csv_cell(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, sort_keys=True, default=str)
    return value


def _read_result_csv(path=RESULTS_CSV):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def _result_sort_key(row):
    """Canonical CSV order used by this notebook."""
    key = str(row.get("case_key", ""))
    size = _as_float(row.get("size"))
    size_key = float("inf") if size is None else size
    if key.startswith("dv:theta:"):
        return (0, 0, size_key, key)
    if key.startswith("lllv:cycle:"):
        return (1, 0, size_key, key)
    if key.startswith("lllv:theta:"):
        return (1, 1, size_key, key)
    if key.startswith("lllv:bouquet:"):
        return (1, 2, size_key, key)
    return (9, 9, size_key, key)


def _write_result_csv(rows, path=RESULTS_CSV):
    path.parent.mkdir(parents=True, exist_ok=True)
    extra = sorted({key for row in rows for key in row if key not in PREFERRED_COLUMNS})
    columns = PREFERRED_COLUMNS + extra
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in sorted(rows, key=_result_sort_key):
            writer.writerow({key: _csv_cell(row.get(key)) for key in columns})
        handle.flush()
        os.fsync(handle.fileno())
    tmp.replace(path)


def _replace_result_row(rows, row):
    # case_key is the physical benchmark identity.  Replacing by case_key,
    # rather than by (settings_hash, case_key), also removes old duplicate
    # rows from earlier benchmark settings as each case is refreshed.
    key = str(row["case_key"])
    kept = [old for old in rows if str(old.get("case_key")) != key]
    kept.append(row)
    _write_result_csv(kept)
    return kept


def _as_float(value):
    if value in (None, ""):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _format_seconds(value):
    value = _as_float(value)
    return "" if value is None else f"{value:.4g}s"


def _kg_result(row):
    return row.get("knottedgraph_result") or row.get("knottedgraph_vs_published") or row.get("knottedgraph_vs_theorem") or ""


def _topoly_result(row):
    return row.get("topoly_result") or row.get("topoly_vs_published") or row.get("topoly_vs_theorem") or ""


def _has_topoly_data(row):
    if not row:
        return False
    return any(
        str(row.get(key, "")).strip()
        for key in (
            "topoly_s",
            "topoly_status",
            "topoly_result",
            "topoly_vs_published",
            "topoly_vs_theorem",
            "topoly_error",
        )
    )


def _cached_topoly_row(rows, case_key):
    """Return the newest duplicate carrying any Topoly information."""
    key = str(case_key)
    for row in reversed(rows):
        if str(row.get("case_key")) == key and _has_topoly_data(row):
            return row
    return None


def _topoly_family_stop_trigger(row):
    """True when this Topoly row should stop larger cases in the same family.

    Benchmark policy: either FAIL or ERROR is a hard cutoff. A row that was
    only skipped because of an earlier cutoff is not itself a new trigger.
    """
    if not row:
        return False
    status = str(row.get("topoly_status", "")).strip().lower()
    if status in {"skipped_after_family_error", "skipped_after_family_nonpass"}:
        return False
    result = str(_topoly_result(row)).strip().upper()
    return result in {"FAIL", "ERROR"}


def _topoly_family_already_blocked(row):
    if not row:
        return False
    status = str(row.get("topoly_status", "")).strip().lower()
    return (
        status in {"skipped_after_family_error", "skipped_after_family_nonpass"}
        or _topoly_family_stop_trigger(row)
    )


def _merge_cached_topoly(row, cached):
    """Keep the old Topoly measurement/result while refreshing KnottedGraph."""
    for key, value in cached.items():
        if key.startswith("topoly_") and key != "topoly_over_knottedgraph":
            row[key] = value
    # Backward-compatible theorem-specific columns are also Topoly cache data.
    for key in ("topoly_vs_published", "topoly_vs_theorem"):
        if key in cached:
            row[key] = cached[key]
    return _refresh_topoly_ratio(row)


def _mark_topoly_skipped_after_family_nonpass(row, *, family, cutoff_size):
    message = (
        "Topoly not executed: an earlier Topoly FAIL or ERROR occurred in "
        f"family {family!r} at size {cutoff_size}; larger sizes are skipped."
    )
    row["topoly_status"] = "skipped_after_family_nonpass"
    row["topoly_result"] = "ERROR"
    row["topoly_error"] = message
    row.pop("topoly_s", None)
    row.pop("topoly_over_knottedgraph", None)
    if row.get("benchmark") == "dobrynin_vesnin_theta_validation":
        row["topoly_vs_published"] = "ERROR"
    elif row.get("benchmark") == "li_lei_li_vesnin_edge_replacement_validation":
        row["topoly_vs_theorem"] = "ERROR"
    return row


def _refresh_topoly_ratio(row):
    kg_s = _as_float(row.get("knottedgraph_s"))
    tp_s = _as_float(row.get("topoly_s"))
    if kg_s is not None and kg_s > 0 and tp_s is not None:
        row["topoly_over_knottedgraph"] = tp_s / kg_s
    else:
        row.pop("topoly_over_knottedgraph", None)
    return row


def _stage_log(case_key, stage, detail=""):
    """Emit an immediately flushed benchmark-stage diagnostic."""
    if not LIVE_STAGE_LOGS:
        return
    suffix = f" | {detail}" if detail else ""
    print(f"[{case_key}] {stage}{suffix}", flush=True)


def _print_case_status(row, *, topoly_mode="run"):
    topoly_note = {
        "cached": " [cached]",
        "blocked": " [family-skipped]",
        "run": "",
    }.get(topoly_mode, "")
    projection_source = str(row.get("projection_source", "") or "")
    projection_note = f" prep={_format_seconds(row.get('projection_s')) or 'n/a'}"
    if projection_source:
        projection_note += f"[{projection_source}]"
    wall_note = f" wall={_format_seconds(row.get('case_wall_s')) or 'n/a'}"
    print(
        f"saved {row.get('case_key', row.get('label', 'case'))}: "
        f"KG {_format_seconds(row.get('knottedgraph_s')) or 'n/a'} "
        f"{_kg_result(row) or row.get('knottedgraph_status', '')}; "
        f"Topoly {_format_seconds(row.get('topoly_s')) or 'n/a'} "
        f"{_topoly_result(row) or row.get('topoly_status', '')}{topoly_note};"
        f"{projection_note}{wall_note}"
    )


def _run_and_capture(run_one):
    """Run a case without hiding live stage diagnostics.

    The older notebook redirected stdout here to suppress internal benchmark
    chatter. That also hid the exact stage at which a large case was spending
    time. Live diagnostics take priority in this scaling notebook.
    """
    return run_one()


def _normalise_dv_row(raw, *, settings_hash, timeout_s):
    row = dict(raw)
    n = int(row["n"])
    row.update(
        {
            "settings_hash": settings_hash,
            "schema_version": RESULT_SCHEMA_VERSION,
            "benchmark": "dobrynin_vesnin_theta_validation",
            "suite": "Dobrynin-Vesnin Theta(n)",
            "case_key": f"dv:theta:{n}",
            "graph_type": f"dv_{row.get('abstract_graph', 'theta')}",
            "size": n,
            "label": f"Theta({n})",
            "timeout_s": timeout_s,
            "knottedgraph_timeout_s": row.get("knottedgraph_timeout_s", timeout_s),
            "topoly_timeout_s": row.get("topoly_timeout_s", ""),
            "knottedgraph_result": row.get("knottedgraph_vs_published", ""),
            "topoly_result": row.get("topoly_vs_published", ""),
        }
    )
    return row


def _normalise_lllv_row(raw, *, settings_hash, timeout_s):
    row = dict(raw)
    size = int(row["size"])
    row.update(
        {
            "settings_hash": settings_hash,
            "schema_version": RESULT_SCHEMA_VERSION,
            "benchmark": "li_lei_li_vesnin_edge_replacement_validation",
            "suite": "Li-Lei-Li-Vesnin infinity_+ edge replacement",
            "case_key": f"lllv:{row['family']}:{size}:infinity_plus",
            "graph_type": row["family"],
            "size": size,
            "timeout_s": timeout_s,
            "knottedgraph_timeout_s": row.get("knottedgraph_timeout_s", timeout_s),
            "topoly_timeout_s": row.get("topoly_timeout_s", ""),
            "knottedgraph_result": row.get("knottedgraph_vs_theorem", ""),
            "topoly_result": row.get("topoly_vs_theorem", ""),
        }
    )
    return row


def _error_row(*, settings_hash, benchmark, suite, case_key, graph_type, size, label, timeout_s, elapsed_s, exc, **extra):
    row = {
        "settings_hash": settings_hash,
        "schema_version": RESULT_SCHEMA_VERSION,
        "benchmark": benchmark,
        "suite": suite,
        "case_key": case_key,
        "graph_type": graph_type,
        "size": size,
        "label": label,
        "timeout_s": timeout_s,
        "case_wall_s": elapsed_s,
        "knottedgraph_result": "ERROR",
        # Do not label Topoly ERROR here: this exception happened outside the
        # framework result wrapper, so it is not evidence that Topoly itself
        # failed.  A cached/blocked Topoly state is merged below by the runner.
        "row_error": f"{type(exc).__name__}: {exc}",
    }
    row.update(extra)
    return row


def _run_cases_to_csv(
    cases,
    *,
    settings_hash,
    run_case,
    case_key,
    case_family,
    case_size,
    normalise_row,
    error_row,
    desc,
):
    """Run KnottedGraph for every case while incrementally caching Topoly.

    Rules:
      * KnottedGraph is ALWAYS executed, even when the case already exists.
      * If any Topoly data already exists for a case, it is copied verbatim
        from the CSV and Topoly is not called again.
      * After the first Topoly FAIL or ERROR in a family, Topoly is skipped
        for all larger uncached cases in that family and ERROR is written.
      * Rows are replaced by case_key, so resumed runs do not create duplicates.
    """
    rows = _read_result_csv()
    if RESET_CURRENT_RESULTS:
        rows = [row for row in rows if str(row.get("settings_hash")) != str(settings_hash)]
        _write_result_csv(rows)

    # Infer existing family cutoffs before running.  This makes resumption obey
    # a Topoly failure already present in the CSV instead of restarting Topoly.
    family_nonpass_cutoff = {}
    for case in cases:
        cached = _cached_topoly_row(rows, case_key(case))
        if cached is None or not _topoly_family_already_blocked(cached):
            continue
        family = str(case_family(case))
        size = float(case_size(case))
        old = family_nonpass_cutoff.get(family)
        if old is None or size < old:
            family_nonpass_cutoff[family] = size

    output = []
    cached_count = sum(
        _cached_topoly_row(rows, case_key(case)) is not None for case in cases
    )
    print(f"CSV: {RESULTS_CSV}")
    print(
        f"Topoly cache: {cached_count}/{len(cases)} requested cases already have "
        "Topoly data; KnottedGraph will still rerun for every case."
    )
    if family_nonpass_cutoff:
        print("Existing Topoly FAIL/ERROR family cutoffs:", family_nonpass_cutoff)

    for case in tqdm(cases, desc=desc):
        key = str(case_key(case))
        family = str(case_family(case))
        size = float(case_size(case))
        cached_topoly = (
            _cached_topoly_row(rows, key)
            if RESUME_FROM_CSV and REUSE_TOPOLY_FROM_CSV
            else None
        )
        cutoff = family_nonpass_cutoff.get(family)
        blocked = (
            STOP_TOPOLY_AFTER_FAMILY_NONPASS
            and cutoff is not None
            and size >= cutoff
        )

        # Cached Topoly always wins, including cached PASS/FAIL/ERROR/skipped
        # rows. Otherwise Topoly runs only if this family has not been blocked.
        run_topoly = cached_topoly is None and not blocked
        topoly_mode = "run" if run_topoly else ("cached" if cached_topoly is not None else "blocked")

        _stage_log(
            key,
            "CASE BEGIN",
            f"family={family} size={size:g} Topoly={topoly_mode}",
        )
        start = time.perf_counter()
        try:
            raw = _run_and_capture(
                lambda case=case, run_topoly=run_topoly: run_case(case, run_topoly)
            )
            row = normalise_row(raw)
        except Exception as exc:
            row = error_row(case, time.perf_counter() - start, exc)

        # Merge Topoly state after the KnottedGraph run (or even after a row-
        # level exception) so an existing Topoly measurement is never lost.
        if cached_topoly is not None:
            row = _merge_cached_topoly(row, cached_topoly)
        elif blocked:
            row = _mark_topoly_skipped_after_family_nonpass(
                row, family=family, cutoff_size=cutoff
            )
        else:
            row = _refresh_topoly_ratio(row)

        # A newly observed Topoly execution failure establishes the cutoff for
        # every later uncached size in the same family.  The current error row
        # itself is preserved as the actual Topoly error.
        if run_topoly and STOP_TOPOLY_AFTER_FAMILY_NONPASS and _topoly_family_stop_trigger(row):
            old = family_nonpass_cutoff.get(family)
            if old is None or size < old:
                family_nonpass_cutoff[family] = size
                print(
                    f"Topoly family stop: {family} returned {_topoly_result(row)} at size {size:g}; "
                    "later uncached Topoly cases in this family will be skipped."
                )

        row["case_wall_s"] = time.perf_counter() - start
        _stage_log(
            key,
            "CSV SAVE",
            f"case_wall={row['case_wall_s']:.3f}s",
        )
        rows = _replace_result_row(rows, row)
        output.append(row)
        _print_case_status(row, topoly_mode=topoly_mode)

    return output

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error

print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())
print("Results CSV:", RESULTS_CSV)
print("Projection cache:", PROJECTION_CACHE_DIR)
print("LLLV infinity_+ samples per replacement:", LLLV_INFINITY_SAMPLES)
if not native_available():
    raise RuntimeError(
        "This benchmark requires the compiled native Yamada backend. "
        "Install the checkout with `python -m pip install -e .` in the active environment."
    )

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install benchmark dependencies with: python -m pip install -e '.[benchmark]'") from exc
print("Topoly:", Path(topoly.__file__).resolve())


## 2. Choose benchmark ranges and timeouts

Edit these lists when you want a smaller smoke test or a longer scaling run. The large values are useful for paper figures but may take much longer than a tutorial run.


In [ ]:
# Input selection. Edit these lists to control the extent of each infinite family.
THETA_N_VALUES = list(range(3, 20))+ [50, 100, 200, 300, 400, 500]
# Separate timeouts: KG should be fast; Topoly may legitimately take hours.
THETA_KG_TIMEOUT_S = 600.0
THETA_TOPOLY_TIMEOUT_S = 1e5
LLLV_CYCLE_N_VALUES = list(range(3, 20)) + [50, 100, 200, 300, 400, 500]
LLLV_THETA_S_VALUES = list(range(3, 20))  + [50, 100, 200, 300, 400, 500]
LLLV_BOUQUET_Q_VALUES = []
LLLV_REPLACEMENT = "infinity_plus"
LLLV_KG_TIMEOUT_S = 600.0
LLLV_TOPOLY_TIMEOUT_S = 1e5

RUN_DOBRYNIN_VESNIN = True
RUN_LI_LEI_LI_VESNIN = True

assert all(n >= 0 for n in THETA_N_VALUES)
assert all(n >= 3 for n in LLLV_CYCLE_N_VALUES)
assert all(s >= 2 for s in LLLV_THETA_S_VALUES)
assert all(q >= 1 for q in LLLV_BOUQUET_Q_VALUES)

print("Dobrynin-Vesnin Theta(n) values:", THETA_N_VALUES)
print("LLLV reference replacement:", LLLV_REPLACEMENT)
print("Cycle C_n(infinity_+) values:", LLLV_CYCLE_N_VALUES)
print("Theta_s(infinity_+) values:", LLLV_THETA_S_VALUES)
print("Bouquet B_q(infinity_+) values:", LLLV_BOUQUET_Q_VALUES)
print("Results CSV:", RESULTS_CSV)

print("DV KG/Topoly timeouts:", THETA_KG_TIMEOUT_S, THETA_TOPOLY_TIMEOUT_S)
print("LLLV KG/Topoly timeouts:", LLLV_KG_TIMEOUT_S, LLLV_TOPOLY_TIMEOUT_S)


## 5. Topoly interpretation note

Topoly documents `max_cross=15` as the default computational cutoff for polynomial evaluation. In the tested Topoly 1.1.0 path, supplying a larger value through `YamadaGraph.point(max_cross=...)` does not propagate that value through all recursive Yamada skein calls, so larger diagrams can still hit the default limit internally.

For the Dobrynin--Vesnin family in this benchmark, Topoly agrees with the published reference through $n=16$, returns a polynomial at $n=17$ that is not equivalent to the published reference polynomial, and fails to complete the $n=18$ and $n=19$ evaluations. KnottedGraph reproduces the published polynomials for the tested cases in this notebook.


In [ ]:
# Shared paper-reference embedding/oracle implementation.
# Embedded directly so notebooks 03 and 06 remain self-contained on this branch.
from __future__ import annotations

from dataclasses import dataclass
import hashlib
import json
import math
import pickle
import re
from types import SimpleNamespace
from typing import Iterable

import networkx as nx
import numpy as np
import sympy as sp

A = sp.Symbol("A")
SIGMA = A + 1 + A**-1

EMBEDDING_VARIANT = "paper_reference_v3_compact_projection"
REPLACEMENT_NAME = "infinity_plus"

DV_PAPER = (
    "A.A. Dobrynin and A.Yu. Vesnin, The Yamada polynomial for graphs, "
    "embedded knot-wise into three-dimensional space, Vychisl. Sistemy 155 "
    "(1996) 37-86, Theorem 2"
)
DV_FORMULA = (
    "R(Theta(n))(A)=(A^2+A+1+A^-1+A^-2)A^n"
    "-(A+A^-1)A^(-2n)"
    "-(A^2+1+A^-2)(-1)^n A^(-n)"
)
DV_REFERENCE_FIGURE = (
    "Dobrynin-Vesnin (1996), Fig. 6: canonical Theta(n) family from the "
    "two-strand (2,n) torus diagram plus one exterior arc"
)

LLLV_PAPER = (
    "S. Li, F. Lei, X. Li and A.Yu. Vesnin, On Yamada polynomial of spatial "
    "graphs obtained by edge replacements, J. Knot Theory Ramifications 27 "
    "(2018), 1843003; arXiv:1801.09075"
)
LLLV_THEOREM = "Theorem 5.1 / Corollary 5.2 / Example 5.4"
LLLV_FORMULAS = {
    "cycle": "R(C_n(infinity_+))=(-A^-2 sigma)^n + sigma(A^-2+1)^n",
    "theta": (
        "R(Theta_s(infinity_+))=[(-sigma)^s + "
        "sigma(((sigma+1)A^-2+1)^s)]/(1+sigma)"
    ),
    "bouquet": "R(B_q(infinity_+))=(-1)^(q-1) sigma^q",
}
LLLV_REFERENCE_FIGURES = {
    "cycle": (
        "Li-Lei-Li-Vesnin Example 5.4; explicit C_4(infinity_+) drawing in "
        "the follow-up density paper, Fig. 5"
    ),
    "theta": (
        "Li-Lei-Li-Vesnin Example 5.4; explicit Theta_3(infinity_+) drawing "
        "in the follow-up density paper, Fig. 6"
    ),
    "bouquet": "Li-Lei-Li-Vesnin Example 5.4 (B_q(infinity_+) formula)",
}


def _as3(xy: np.ndarray, z: float | np.ndarray = 0.0) -> np.ndarray:
    xy = np.asarray(xy, dtype=float)
    if xy.ndim != 2 or xy.shape[1] != 2:
        raise ValueError("xy must have shape (N,2)")
    zcol = np.broadcast_to(np.asarray(z, dtype=float), (len(xy),))
    return np.column_stack([xy, zcol])


def _bezier_cubic(p0, p1, p2, p3, samples: int = 80) -> np.ndarray:
    p0, p1, p2, p3 = [np.asarray(p, dtype=float) for p in (p0, p1, p2, p3)]
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def _join_paths(*paths: np.ndarray) -> np.ndarray:
    out = []
    for path in paths:
        path = np.asarray(path, dtype=float)
        if not len(path):
            continue
        if out and np.allclose(out[-1][-1], path[0]):
            path = path[1:]
        if len(path):
            out.append(path)
    if not out:
        return np.empty((0, 3), dtype=float)
    return np.vstack(out)


def _line(a, b, samples: int = 2) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (1.0 - t) * a + t * b


def published_dv_theta_terms(n: int) -> dict[int, int]:
    n = int(n)
    if n < 0:
        raise ValueError("Dobrynin-Vesnin Theta(n) requires n >= 0")
    terms: dict[int, int] = {}

    def add(power: int, coefficient: int) -> None:
        terms[power] = terms.get(power, 0) + coefficient
        if terms[power] == 0:
            del terms[power]

    for offset in (2, 1, 0, -1, -2):
        add(n + offset, 1)
    add(-2 * n + 1, -1)
    add(-2 * n - 1, -1)
    third = -((-1) ** n)
    for offset in (2, 0, -2):
        add(-n + offset, third)
    return terms


def reference_dv_theta_graph(n: int, samples_per_crossing: int = 17):
    """Reconstruct the paper's canonical Theta(n) diagram.

    The two actual graph vertices sit at the left.  A small exterior arc joins
    them, while the other two graph edges are the canonical two-strand (2,n)
    diagram closed by the long upper/lower arcs.  Thus the visible diagram has
    exactly the same parity change as the paper: odd n is a theta graph and
    even n is a pince-nez/handcuff graph.
    """
    n = int(n)
    samples_per_crossing = int(samples_per_crossing)
    if n < 0:
        raise ValueError("n must be >= 0")
    if samples_per_crossing < 6:
        raise ValueError("samples_per_crossing must be >= 6")

    y0 = 0.56
    U = np.array([0.0, y0, 0.0])
    V = np.array([0.0, -y0, 0.0])
    x_left = 0.62
    x_right = 2.05 + 0.72 * max(n, 1)
    depth = 0.24

    # Piecewise-linear two-strand braid: each of the n braid cells contributes
    # one projected crossing.  Alternating z signs reproduce the usual braid
    # over/under sequence rather than merely drawing crossings in 2D.
    if n == 0:
        t = np.linspace(0.0, 1.0, 24)
        y_a = np.full_like(t, y0)
        z_a = np.zeros_like(t)
    else:
        count = n * samples_per_crossing + 1
        t = np.linspace(0.0, 1.0, count)
        knots = np.linspace(0.0, 1.0, n + 1)
        knot_values = y0 * ((-1.0) ** np.arange(n + 1))
        y_a = np.interp(t, knots, knot_values)
        z_a = depth * np.sin(np.pi * n * t)
    x = x_left + (x_right - x_left) * t
    braid_a = np.column_stack([x, y_a, z_a])
    braid_b = np.column_stack([x, -y_a, -z_a])

    top_right = np.array([x_right, y0, 0.0])
    bottom_right = np.array([x_right, -y0, 0.0])
    outer_y = 1.16
    top_return = _bezier_cubic(
        top_right,
        [x_right + 0.46, outer_y, 0.0],
        [0.15, outer_y, 0.0],
        U,
        samples=max(70, 8 * max(n, 1)),
    )
    bottom_return = _bezier_cubic(
        bottom_right,
        [x_right + 0.46, -outer_y, 0.0],
        [0.15, -outer_y, 0.0],
        V,
        samples=max(70, 8 * max(n, 1)),
    )

    phi = np.linspace(np.pi / 2, 3 * np.pi / 2, 70)
    exterior = np.column_stack([
        -0.67 * np.cos(phi - np.pi),
        y0 * np.sin(phi),
        np.zeros_like(phi),
    ])
    # Force exact graph-vertex coordinates; the analytic parameterization above
    # is intentionally shaped to the left, but endpoint equality matters to PDCode.
    exterior[0] = U
    exterior[-1] = V

    graph = nx.MultiGraph()
    graph.add_node("u", pos=U.copy())
    graph.add_node("v", pos=V.copy())

    left_u_to_braid = _line(U, braid_a[0], 14)
    left_v_to_braid = _line(V, braid_b[0], 14)

    if n % 2:
        # A: U -> braid -> lower return -> V.
        edge_a = _join_paths(left_u_to_braid, braid_a, bottom_return)
        # B: U -> upper return (reversed) -> braid B (reversed) -> V.
        edge_b = _join_paths(top_return[::-1], braid_b[::-1], left_v_to_braid[::-1])
        graph.add_edge("u", "v", pts=edge_a, role="torus_a")
        graph.add_edge("u", "v", pts=edge_b, role="torus_b")
        graph.add_edge("u", "v", pts=exterior, role="exterior_arc")
        abstract_type = "theta"
    else:
        loop_u = _join_paths(left_u_to_braid, braid_a, top_return)
        loop_v = _join_paths(left_v_to_braid, braid_b, bottom_return)
        graph.add_edge("u", "u", pts=loop_u, role="torus_component_u")
        graph.add_edge("v", "v", pts=loop_v, role="torus_component_v")
        graph.add_edge("u", "v", pts=exterior, role="exterior_arc")
        abstract_type = "handcuff"

    graph.graph.update(
        reference_embedding_variant=EMBEDDING_VARIANT,
        embedding_source=DV_PAPER,
        reference_figure=DV_REFERENCE_FIGURE,
        expected_projected_crossings=n,
        dobrynin_vesnin_abstract_type=abstract_type,
    )
    return graph


def _resample_polyline(points: np.ndarray, samples: int = 121) -> np.ndarray:
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] != 3 or len(points) < 2:
        raise ValueError("points must have shape (N,3), N>=2")
    delta = np.diff(points, axis=0)
    lengths = np.linalg.norm(delta, axis=1)
    cumulative = np.concatenate([[0.0], np.cumsum(lengths)])
    total = cumulative[-1]
    if total <= 1e-12:
        raise ValueError("base path has zero length")
    target = np.linspace(0.0, total, int(samples))
    out = np.empty((len(target), 3), dtype=float)
    for dim in range(3):
        out[:, dim] = np.interp(target, cumulative, points[:, dim])
    return out


def _infinity_plus_pair(
    base_path: np.ndarray,
    *,
    width: float,
    depth: float,
    samples: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Replace one base edge by the paper's infinity_+ two-edge diagram."""
    if samples is None:
        samples = int(LLLV_INFINITY_SAMPLES)
    samples = int(samples)
    if samples < 8 or samples % 2:
        raise ValueError("LLLV_INFINITY_SAMPLES must be an even integer >= 8")
    base = _resample_polyline(base_path, samples=samples)
    xy = base[:, :2]
    tangent = np.gradient(xy, axis=0)
    norm = np.linalg.norm(tangent, axis=1)
    norm[norm < 1e-12] = 1.0
    tangent = tangent / norm[:, None]
    normal = np.column_stack([-tangent[:, 1], tangent[:, 0]])

    t = np.linspace(0.0, 1.0, len(base))
    # Figure-eight lateral displacement.  The sin(pi t) envelope makes the two
    # edges merge cleanly at the genuine graph vertices and cross once at t=1/2.
    # IMPORTANT: the configured value is even, so t=1/2 is
    # not itself a sampled polyline vertex. PDCode intentionally ignores
    # endpoint-only segment touches, therefore the intended crossing must lie
    # in the interior of the two intersecting segments.
    lateral = np.sin(2.0 * np.pi * t) * np.sin(np.pi * t)
    z_profile = np.exp(-((t - 0.5) / 0.070) ** 2) * np.sin(np.pi * t) ** 2

    over = base.copy()
    under = base.copy()
    over[:, :2] += float(width) * lateral[:, None] * normal
    under[:, :2] -= float(width) * lateral[:, None] * normal
    over[:, 2] += float(depth) * z_profile
    under[:, 2] -= float(depth) * z_profile
    over[0] = under[0] = base[0]
    over[-1] = under[-1] = base[-1]
    return over, under


def _add_infinity_replacement(
    graph: nx.MultiGraph,
    u,
    v,
    base_path: np.ndarray,
    *,
    base_edge_index: int,
    width: float,
    depth: float,
) -> None:
    over, under = _infinity_plus_pair(base_path, width=width, depth=depth)
    common = {
        "replacement_piece": REPLACEMENT_NAME,
        "base_edge_index": int(base_edge_index),
    }
    graph.add_edge(
        u,
        v,
        pts=over,
        role="infinity_plus_over",
        crossing_role="over",
        **common,
    )
    graph.add_edge(
        u,
        v,
        pts=under,
        role="infinity_plus_under",
        crossing_role="under",
        **common,
    )


def reference_lllv_cycle_graph(n: int) -> nx.MultiGraph:
    """Paper-faithful C_n(infinity_+) edge-replacement family."""
    n = int(n)
    if n < 3:
        raise ValueError("cycle requires n >= 3")
    radius = 2.05
    # For n=4 this is a literal square with horizontal/vertical sides as in the
    # published C_4(infinity_+) figure.
    angles = np.pi / 4 + 2.0 * np.pi * np.arange(n) / n
    positions = {
        k: np.array([radius * np.cos(a), radius * np.sin(a), 0.0])
        for k, a in enumerate(angles)
    }
    graph = nx.MultiGraph()
    for k, pos in positions.items():
        graph.add_node(k, pos=pos.copy())
    side = float(np.linalg.norm(positions[1] - positions[0]))
    width = min(0.26, 0.13 * side)
    depth = min(0.18, 0.085 * side)
    for k in range(n):
        j = (k + 1) % n
        base = _line(positions[k], positions[j], 2)
        _add_infinity_replacement(
            graph, k, j, base,
            base_edge_index=k, width=width, depth=depth,
        )
    graph.graph.update(
        reference_embedding_variant=EMBEDDING_VARIANT,
        embedding_source=LLLV_PAPER,
        reference_figure=LLLV_REFERENCE_FIGURES["cycle"],
        replacement=REPLACEMENT_NAME,
        lllv_family="cycle",
        lllv_size=n,
        expected_projected_crossings=n,
    )
    return graph


def reference_lllv_theta_graph(s: int) -> nx.MultiGraph:
    """Paper-faithful Theta_s(infinity_+) edge-replacement family."""
    s = int(s)
    if s < 2:
        raise ValueError("theta requires s >= 2")
    U = np.array([-2.25, 0.0, 0.0])
    V = np.array([2.25, 0.0, 0.0])
    graph = nx.MultiGraph()
    graph.add_node("u", pos=U.copy())
    graph.add_node("v", pos=V.copy())

    amplitudes = np.linspace(1.46, -1.46, s)
    spacing = 2.92 / max(s - 1, 1)
    width = min(0.20, max(0.0015, 0.14 * spacing))
    depth = min(0.16, 0.72 * width)
    t = np.linspace(0.0, 1.0, 181)
    x = U[0] + (V[0] - U[0]) * t
    for index, amp in enumerate(amplitudes):
        y = amp * np.sin(np.pi * t)
        base = np.column_stack([x, y, np.zeros_like(x)])
        base[0] = U
        base[-1] = V
        _add_infinity_replacement(
            graph, "u", "v", base,
            base_edge_index=index, width=width, depth=depth,
        )
    graph.graph.update(
        reference_embedding_variant=EMBEDDING_VARIANT,
        embedding_source=LLLV_PAPER,
        reference_figure=LLLV_REFERENCE_FIGURES["theta"],
        replacement=REPLACEMENT_NAME,
        lllv_family="theta",
        lllv_size=s,
        expected_projected_crossings=s,
    )
    return graph


def reference_lllv_bouquet_graph(q: int) -> nx.MultiGraph:
    """B_q(infinity_+) in a separated radial-petal embedding."""
    q = int(q)
    if q < 1:
        raise ValueError("bouquet requires q >= 1")
    O = np.array([0.0, 0.0, 0.0])
    graph = nx.MultiGraph()
    graph.add_node("v", pos=O.copy())
    radius = 2.0
    sector = 2.0 * np.pi / q
    half_sweep = min(0.84, 0.34 * sector)
    width = min(0.11, 0.075 * radius * min(1.0, sector))
    depth = 0.75 * width
    t = np.linspace(0.0, 1.0, 181)
    radial = radius * np.sin(np.pi * t)
    for index in range(q):
        theta0 = 2.0 * np.pi * index / q
        angle = theta0 + half_sweep * np.sin(2.0 * np.pi * t)
        base = np.column_stack([
            radial * np.cos(angle),
            radial * np.sin(angle),
            np.zeros_like(t),
        ])
        base[0] = O
        base[-1] = O
        _add_infinity_replacement(
            graph, "v", "v", base,
            base_edge_index=index, width=width, depth=depth,
        )
    graph.graph.update(
        reference_embedding_variant=EMBEDDING_VARIANT,
        embedding_source=LLLV_PAPER,
        reference_figure=LLLV_REFERENCE_FIGURES["bouquet"],
        replacement=REPLACEMENT_NAME,
        lllv_family="bouquet",
        lllv_size=q,
        expected_projected_crossings=q,
    )
    return graph


def build_lllv_reference_graph(family: str, size: int) -> nx.MultiGraph:
    family = str(family).lower()
    if family == "cycle":
        return reference_lllv_cycle_graph(size)
    if family == "theta":
        return reference_lllv_theta_graph(size)
    if family == "bouquet":
        return reference_lllv_bouquet_graph(size)
    raise ValueError(f"unknown LLLV reference family {family!r}")


def _add_terms(target: dict[int, int], source: dict[int, int], scale: int = 1) -> None:
    for power, coefficient in source.items():
        value = target.get(power, 0) + int(scale) * int(coefficient)
        if value:
            target[power] = value
        elif power in target:
            del target[power]


def _sigma_power(n: int) -> dict[int, int]:
    """Exact coefficients of (A + 1 + A^-1)^n in O(n^2) integer additions."""
    n = int(n)
    coeff = [1]
    for _ in range(n):
        nxt = [0] * (len(coeff) + 2)
        for i, value in enumerate(coeff):
            nxt[i] += value
            nxt[i + 1] += value
            nxt[i + 2] += value
        coeff = nxt
    return {(-n + i): int(value) for i, value in enumerate(coeff) if value}


def _h_infinity_power(n: int) -> dict[int, int]:
    """Power of H=1+A^-1+2A^-2+A^-3 used in Example 5.4."""
    n = int(n)
    coeff = [1]  # index j is coefficient of x^j with x=A^-1
    base = (1, 1, 2, 1)
    for _ in range(n):
        nxt = [0] * (len(coeff) + 3)
        for i, value in enumerate(coeff):
            if not value:
                continue
            for j, base_value in enumerate(base):
                nxt[i + j] += value * base_value
        coeff = nxt
    return {-i: int(value) for i, value in enumerate(coeff) if value}


def _times_sigma(terms: dict[int, int]) -> dict[int, int]:
    out: dict[int, int] = {}
    for power, coefficient in terms.items():
        for shift in (-1, 0, 1):
            out[power + shift] = out.get(power + shift, 0) + coefficient
    return {power: coefficient for power, coefficient in out.items() if coefficient}


def _laurent_divide_exact(
    numerator: dict[int, int],
    denominator: dict[int, int],
) -> dict[int, int]:
    """Exact Laurent division for a monic highest-power denominator."""
    remaining = {int(p): int(c) for p, c in numerator.items() if c}
    denominator = {int(p): int(c) for p, c in denominator.items() if c}
    if not denominator:
        raise ZeroDivisionError("empty Laurent denominator")
    dmax = max(denominator)
    lead = denominator[dmax]
    if lead not in (1, -1):
        raise ValueError("division helper expects unit leading coefficient")
    quotient: dict[int, int] = {}
    # A divisible Laurent polynomial of width W needs at most O(W) eliminations.
    budget = 4 * (max(remaining) - min(remaining) + len(denominator) + 4) if remaining else 0
    steps = 0
    while remaining:
        steps += 1
        if steps > budget:
            raise ValueError("Laurent expression did not divide exactly")
        rmax = max(remaining)
        qpower = rmax - dmax
        rlead = remaining[rmax]
        if rlead % lead:
            raise ValueError("non-integral Laurent quotient")
        qcoeff = rlead // lead
        quotient[qpower] = quotient.get(qpower, 0) + qcoeff
        for dpower, dcoeff in denominator.items():
            power = qpower + dpower
            value = remaining.get(power, 0) - qcoeff * dcoeff
            if value:
                remaining[power] = value
            elif power in remaining:
                del remaining[power]
    return {power: coefficient for power, coefficient in quotient.items() if coefficient}


def published_lllv_terms(family: str, size: int) -> dict[int, int]:
    """Exact Example-5.4 oracle without symbolic expansion bottlenecks.

    The previous SymPy-power implementation becomes unnecessarily expensive for
    the notebook's n=500 cases.  These recurrences only manipulate the integer
    Laurent coefficients and therefore keep theorem-oracle time negligible
    relative to the timed Yamada evaluations.
    """
    family = str(family).lower()
    size = int(size)
    if family == "cycle":
        if size < 3:
            raise ValueError("cycle requires n >= 3")
        out: dict[int, int] = {}
        sigma_n = _sigma_power(size)
        sign = -1 if size % 2 else 1
        # (-A^-2 sigma)^n
        _add_terms(out, {power - 2 * size: coefficient for power, coefficient in sigma_n.items()}, sign)
        # + sigma (1+A^-2)^n
        binomial = {-2 * k: math.comb(size, k) for k in range(size + 1)}
        _add_terms(out, _times_sigma(binomial))
        return out

    if family == "theta":
        if size < 2:
            raise ValueError("theta requires s >= 2")
        numerator: dict[int, int] = {}
        _add_terms(numerator, _sigma_power(size), -1 if size % 2 else 1)
        _add_terms(numerator, _times_sigma(_h_infinity_power(size)))
        # 1+sigma = A^-1 + 2 + A
        return _laurent_divide_exact(numerator, {-1: 1, 0: 2, 1: 1})

    if family == "bouquet":
        if size < 1:
            raise ValueError("bouquet requires q >= 1")
        sign = -1 if (size - 1) % 2 else 1
        return {power: sign * coefficient for power, coefficient in _sigma_power(size).items()}

    raise ValueError(f"unknown LLLV family {family!r}")

def _embedding_hash(graph: nx.MultiGraph) -> str:
    payload = []
    for node, data in sorted(graph.nodes(data=True), key=lambda item: repr(item[0])):
        payload.append(("node", repr(node), np.asarray(data.get("pos", []), dtype=float).round(12).tolist()))
    edges = []
    for u, v, key, data in graph.edges(keys=True, data=True):
        pts = np.asarray(data.get("pts", []), dtype=float).round(12).tolist()
        edges.append((repr(u), repr(v), repr(key), data.get("role"), pts))
    payload.extend(("edge", *entry) for entry in sorted(edges, key=lambda item: (item[0], item[1], item[2])))
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def _projection_cache_path(
    *,
    case_key: str,
    embedding_hash: str,
    expected_crossings: int,
) -> tuple[Path, str]:
    """Return deterministic cache path/key for one exact geometric embedding."""
    payload = {
        "schema": int(PROJECTION_CACHE_SCHEMA),
        "embedding_variant": EMBEDDING_VARIANT,
        "case_key": str(case_key),
        "embedding_hash": str(embedding_hash),
        "expected_crossings": int(expected_crossings),
        "lllv_infinity_samples": int(LLLV_INFINITY_SAMPLES),
    }
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    cache_key = hashlib.sha256(encoded).hexdigest()
    safe_case = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(case_key)).strip("_") or "case"
    return PROJECTION_CACHE_DIR / f"{safe_case}__{cache_key[:20]}.pkl", cache_key


def _processor_from_cache_payload(payload: dict):
    """Minimal processor-like object required by benchmark _run_with_timeout."""
    return SimpleNamespace(
        vertices=dict(payload["vertices"]),
        crossings=dict(payload["crossings"]),
        arcs=dict(payload["arcs"]),
    )


def _load_projection_cache(
    path: Path,
    *,
    cache_key: str,
    embedding_hash: str,
    expected_crossings: int,
):
    if not REUSE_PROJECTION_CACHE or not path.exists():
        return None
    try:
        with path.open("rb") as handle:
            payload = pickle.load(handle)
        if int(payload.get("schema", -1)) != int(PROJECTION_CACHE_SCHEMA):
            return None
        if str(payload.get("cache_key", "")) != str(cache_key):
            return None
        if str(payload.get("embedding_hash", "")) != str(embedding_hash):
            return None
        if int(payload.get("expected_crossings", -1)) != int(expected_crossings):
            return None
        pdcode = str(payload["pdcode"])
        if hashlib.sha256(pdcode.encode()).hexdigest() != str(payload.get("pd_hash", "")):
            return None
        processor = _processor_from_cache_payload(payload)
        if len(processor.crossings) != int(expected_crossings):
            return None
        return processor, pdcode
    except Exception as exc:
        # A broken/stale cache is never allowed to invalidate the benchmark.
        print(f"Projection cache ignored ({path.name}): {type(exc).__name__}: {exc}")
        return None


def _write_projection_cache(
    path: Path,
    *,
    cache_key: str,
    embedding_hash: str,
    expected_crossings: int,
    processor,
    pdcode: str,
):
    if not WRITE_PROJECTION_CACHE:
        return
    payload = {
        "schema": int(PROJECTION_CACHE_SCHEMA),
        "cache_key": str(cache_key),
        "embedding_variant": EMBEDDING_VARIANT,
        "embedding_hash": str(embedding_hash),
        "expected_crossings": int(expected_crossings),
        "lllv_infinity_samples": int(LLLV_INFINITY_SAMPLES),
        "pdcode": str(pdcode),
        "pd_hash": hashlib.sha256(pdcode.encode()).hexdigest(),
        # Store the exact combinatorial objects consumed by the timed worker.
        # These objects already have to be spawn-pickleable in the original
        # benchmark because _run_with_timeout uses multiprocessing("spawn").
        "vertices": dict(processor.vertices),
        "crossings": dict(processor.crossings),
        "arcs": dict(processor.arcs),
    }
    tmp = path.with_suffix(path.suffix + ".tmp")
    try:
        with tmp.open("wb") as handle:
            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
            handle.flush()
            os.fsync(handle.fileno())
        tmp.replace(path)
    except Exception as exc:
        try:
            tmp.unlink(missing_ok=True)
        except Exception:
            pass
        print(f"Projection cache write skipped ({path.name}): {type(exc).__name__}: {exc}")


def prepare_reference_graph(
    graph: nx.MultiGraph,
    expected_crossings: int,
    *,
    case_key: str,
):
    """Prepare exact PDCode, reusing deterministic geometric preprocessing.

    Only projection/preprocessing is cached. KnottedGraph Yamada is never cached.
    """
    from knotted_graph.projection import PDCode

    expected_crossings = int(expected_crossings)

    hash_start = time.perf_counter()
    embedding_hash = _embedding_hash(graph)
    hash_s = time.perf_counter() - hash_start
    _stage_log(
        case_key,
        "EMBEDDING HASH",
        f"{hash_s:.3f}s; V={graph.number_of_nodes()} E={graph.number_of_edges()}",
    )

    cache_path, cache_key = _projection_cache_path(
        case_key=case_key,
        embedding_hash=embedding_hash,
        expected_crossings=expected_crossings,
    )

    start = time.perf_counter()
    _stage_log(case_key, "PDCODE CACHE LOOKUP", cache_path.name)
    cached = _load_projection_cache(
        cache_path,
        cache_key=cache_key,
        embedding_hash=embedding_hash,
        expected_crossings=expected_crossings,
    )
    if cached is not None:
        processor, pdcode = cached
        elapsed = time.perf_counter() - start
        _stage_log(
            case_key,
            "PDCODE CACHE HIT",
            f"{elapsed:.3f}s; crossings={len(processor.crossings)}",
        )
        return graph, processor, pdcode, {
            "projection_source": "cache",
            "projection_s": elapsed,
            "projection_cache_key": cache_key[:20],
            "embedding_hash": embedding_hash,
        }

    _stage_log(
        case_key,
        "PDCODE COMPUTE START",
        f"expected_crossings={expected_crossings}; cache miss",
    )
    processor = PDCode(graph)
    pdcode = processor.compute(rotation_angles=(0.0, 0.0, 0.0))
    actual = len(processor.crossings)
    elapsed = time.perf_counter() - start
    _stage_log(
        case_key,
        "PDCODE COMPUTE END",
        f"{elapsed:.3f}s; detected_crossings={actual}",
    )
    if actual != expected_crossings:
        raise AssertionError(
            f"paper-reference embedding expected {expected_crossings} projected crossings; "
            f"PDCode detected {actual}. If this follows a change to "
            f"LLLV_INFINITY_SAMPLES={LLLV_INFINITY_SAMPLES}, increase that value and rerun."
        )

    cache_write_start = time.perf_counter()
    _write_projection_cache(
        cache_path,
        cache_key=cache_key,
        embedding_hash=embedding_hash,
        expected_crossings=expected_crossings,
        processor=processor,
        pdcode=pdcode,
    )
    _stage_log(
        case_key,
        "PDCODE CACHE WRITE",
        f"{time.perf_counter() - cache_write_start:.3f}s",
    )
    return graph, processor, pdcode, {
        "projection_source": "computed",
        "projection_s": elapsed,
        "projection_cache_key": cache_key[:20],
        "embedding_hash": embedding_hash,
    }



def _run_with_timeout_safe(
    base,
    framework: str,
    processor,
    pdcode: str,
    timeout_s: float,
):
    """Run one framework in a spawned worker without queue/join deadlock.

    Critical ordering:
        child starts
        parent drains Queue
        only then parent joins child

    The old helper joined the child before reading the Queue. If the serialized
    result grew beyond the OS pipe buffer, the child feeder thread could block
    flushing the Queue, while the parent blocked waiting for child exit.
    """
    import multiprocessing as mp
    import queue as queue_module

    context = mp.get_context("spawn")
    result_queue = context.Queue()
    process = context.Process(
        target=base._worker,
        args=(
            framework,
            list(processor.vertices.values()),
            list(processor.crossings.values()),
            list(processor.arcs.values()),
            pdcode,
            result_queue,
        ),
    )

    process.start()
    deadline = time.monotonic() + float(timeout_s)
    result = None

    # Drain the queue while the child is alive. This is the essential fix.
    while True:
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            break

        try:
            result = result_queue.get(timeout=min(0.25, remaining))
            break
        except queue_module.Empty:
            # If the worker exited without producing a message, do not wait for
            # the entire timeout.
            if not process.is_alive():
                break

    if result is not None:
        # The queue payload has already been consumed, so the child feeder
        # thread can finish and process termination should be prompt.
        process.join(5.0)
        if process.is_alive():
            process.terminate()
            process.join(5.0)

        try:
            result_queue.close()
            result_queue.join_thread()
        except Exception:
            pass

        return result

    # No result was received.
    if process.is_alive():
        process.terminate()
        process.join(5.0)
        status = {
            "status": "timeout",
            "framework": framework,
            "timeout_s": float(timeout_s),
        }
    else:
        status = {
            "status": "error",
            "framework": framework,
            "error": (
                f"worker exited with code {process.exitcode} "
                "without returning data"
            ),
        }

    try:
        result_queue.close()
    except Exception:
        pass

    return status


def _run_frameworks(
    processor,
    pdcode: str,
    kg_timeout_s: float,
    topoly_timeout_s: float,
    *,
    run_topoly: bool = True,
    case_key: str = "case",
):
    import benchmark_topoly_paper_scaling as base

    # KnottedGraph is intentionally NEVER cached in this notebook.
    _stage_log(
        case_key,
        "KG WORKER START",
        f"timeout={kg_timeout_s:g}s; crossings={len(processor.crossings)}; queue=read-before-join",
    )
    kg_wall_start = time.perf_counter()
    kg = _run_with_timeout_safe(
        base, "knottedgraph", processor, pdcode, float(kg_timeout_s)
    )
    kg_wall = time.perf_counter() - kg_wall_start
    _stage_log(
        case_key,
        "KG WORKER END",
        f"status={kg.get('status')} worker_time={kg.get('time_s', 'n/a')} "
        f"wall={kg_wall:.3f}s",
    )

    if run_topoly:
        _stage_log(
            case_key,
            "TOPOLY START",
            f"timeout={topoly_timeout_s:g}s",
        )
        tp_wall_start = time.perf_counter()
        tp = _run_with_timeout_safe(
            base, "topoly", processor, pdcode, float(topoly_timeout_s)
        )
        tp_wall = time.perf_counter() - tp_wall_start
        _stage_log(
            case_key,
            "TOPOLY END",
            f"status={tp.get('status')} worker_time={tp.get('time_s', 'n/a')} "
            f"wall={tp_wall:.3f}s",
        )
    else:
        tp = None
        _stage_log(
            case_key,
            "TOPOLY NOT CALLED",
            "cached result or family FAIL/ERROR cutoff",
        )

    return base, kg, tp


def _theorem_verdict(base, result: dict, expected: dict[int, int], *, exact: bool) -> str:
    if result.get("status") != "ok":
        return "ERROR"
    if exact:
        return "PASS" if result.get("terms") == expected else "FAIL"
    try:
        base._validate_laurent_unit(expected, result.get("terms", {}))
    except AssertionError:
        return "FAIL"
    return "PASS"


def compare_dv_theta_reference(
    n: int,
    kg_timeout_s: float,
    topoly_timeout_s: float,
    strict: bool = False,
    *,
    run_topoly: bool = True,
) -> dict:
    n = int(n)
    case_key = f"dv:theta:{n}"
    case_start = time.perf_counter()

    _stage_log(case_key, "GRAPH BUILD START")
    t = time.perf_counter()
    graph = reference_dv_theta_graph(n)
    _stage_log(
        case_key,
        "GRAPH BUILD END",
        f"{time.perf_counter()-t:.3f}s; V={graph.number_of_nodes()} E={graph.number_of_edges()}",
    )

    graph, processor, pdcode, projection = prepare_reference_graph(
        graph, n, case_key=case_key
    )

    _stage_log(case_key, "ORACLE BUILD START", "Dobrynin-Vesnin Theorem 2")
    t = time.perf_counter()
    expected = published_dv_theta_terms(n)
    _stage_log(case_key, "ORACLE BUILD END", f"{time.perf_counter()-t:.3f}s")

    base, kg, tp = _run_frameworks(
        processor,
        pdcode,
        kg_timeout_s,
        topoly_timeout_s,
        run_topoly=run_topoly,
        case_key=case_key,
    )

    _stage_log(case_key, "THEOREM CHECK START")
    t = time.perf_counter()
    kg_verdict = _theorem_verdict(base, kg, expected, exact=True)
    tp_verdict = _theorem_verdict(base, tp, expected, exact=False) if tp is not None else None
    _stage_log(
        case_key,
        "THEOREM CHECK END",
        f"{time.perf_counter()-t:.3f}s; KG={kg_verdict}; Topoly={tp_verdict}",
    )

    if strict and kg_verdict != "PASS":
        raise AssertionError(
            f"KnottedGraph vs published Dobrynin-Vesnin formula: {kg_verdict}"
        )

    row = {
        "n": n,
        "crossings": len(processor.crossings),
        "extra_projection_crossings": len(processor.crossings) - n,
        "abstract_graph": graph.graph["dobrynin_vesnin_abstract_type"],
        "V": graph.number_of_nodes(),
        "E": graph.number_of_edges(),
        "max_degree": max(dict(graph.degree()).values()),
        "paper": DV_PAPER,
        "theorem": "Theorem 2",
        "formula": DV_FORMULA,
        "embedding_variant": EMBEDDING_VARIANT,
        "embedding_source": DV_PAPER,
        "reference_figure": DV_REFERENCE_FIGURE,
        "embedding_hash": projection["embedding_hash"],
        "projection_source": projection["projection_source"],
        "projection_s": projection["projection_s"],
        "projection_cache_key": projection["projection_cache_key"],
        "pd_hash": hashlib.sha256(pdcode.encode()).hexdigest(),
        "case_hash": hashlib.sha256(
            f"{EMBEDDING_VARIANT}:dv:{n}".encode()
        ).hexdigest()[:16],
        "knottedgraph_timeout_s": float(kg_timeout_s),
        "topoly_timeout_s": float(topoly_timeout_s),
        "knottedgraph_status": kg.get("status"),
        "knottedgraph_s": kg.get("time_s"),
        "knottedgraph_vs_published": kg_verdict,
    }
    if tp is not None:
        row.update(
            {
                "topoly_status": tp.get("status"),
                "topoly_s": tp.get("time_s"),
                "topoly_vs_published": tp_verdict,
            }
        )
    if kg.get("status") != "ok":
        row["knottedgraph_error"] = (
            kg.get("error")
            or f"KnottedGraph status: {kg.get('status')}; timeout={kg_timeout_s}s"
        )
    if tp is not None and tp.get("status") != "ok":
        row["topoly_error"] = (
            tp.get("error")
            or f"Topoly status: {tp.get('status')}; timeout={topoly_timeout_s}s"
        )
    if tp is not None and kg.get("time_s") and tp.get("time_s"):
        row["topoly_over_knottedgraph"] = tp["time_s"] / kg["time_s"]

    _stage_log(
        case_key,
        "CASE COMPUTE END",
        f"{time.perf_counter()-case_start:.3f}s before CSV merge/save",
    )
    return row


@dataclass(frozen=True)
class LLLVReferenceCase:
    family: str
    size: int

    @property
    def label(self) -> str:
        if self.family == "cycle":
            return f"C_{self.size}(infinity_+)"
        if self.family == "theta":
            return f"Theta_{self.size}(infinity_+)"
        if self.family == "bouquet":
            return f"B_{self.size}(infinity_+)"
        return f"{self.family}_{self.size}(infinity_+)"

    @property
    def case_key(self) -> str:
        return f"lllv:{self.family}:{self.size}:infinity_plus"


def build_lllv_reference_cases(
    *,
    cycle_n_values: Iterable[int],
    theta_s_values: Iterable[int],
    bouquet_q_values: Iterable[int],
) -> list[LLLVReferenceCase]:
    cases = [LLLVReferenceCase("cycle", int(n)) for n in cycle_n_values]
    cases += [LLLVReferenceCase("theta", int(s)) for s in theta_s_values]
    cases += [LLLVReferenceCase("bouquet", int(q)) for q in bouquet_q_values]
    return cases


def compare_lllv_reference_case(
    case: LLLVReferenceCase,
    kg_timeout_s: float,
    topoly_timeout_s: float,
    strict: bool = False,
    *,
    run_topoly: bool = True,
) -> dict:
    case_key = case.case_key
    case_start = time.perf_counter()

    _stage_log(case_key, "GRAPH BUILD START")
    t = time.perf_counter()
    graph = build_lllv_reference_graph(case.family, case.size)
    _stage_log(
        case_key,
        "GRAPH BUILD END",
        f"{time.perf_counter()-t:.3f}s; V={graph.number_of_nodes()} E={graph.number_of_edges()}",
    )

    expected_crossings = case.size
    graph, processor, pdcode, projection = prepare_reference_graph(
        graph, expected_crossings, case_key=case_key
    )

    _stage_log(
        case_key,
        "ORACLE BUILD START",
        f"LLLV theorem family={case.family} size={case.size}",
    )
    t = time.perf_counter()
    expected = published_lllv_terms(case.family, case.size)
    _stage_log(case_key, "ORACLE BUILD END", f"{time.perf_counter()-t:.3f}s")

    base, kg, tp = _run_frameworks(
        processor,
        pdcode,
        kg_timeout_s,
        topoly_timeout_s,
        run_topoly=run_topoly,
        case_key=case_key,
    )

    _stage_log(case_key, "THEOREM CHECK START")
    t = time.perf_counter()
    kg_verdict = _theorem_verdict(base, kg, expected, exact=True)
    tp_verdict = _theorem_verdict(base, tp, expected, exact=False) if tp is not None else None
    _stage_log(
        case_key,
        "THEOREM CHECK END",
        f"{time.perf_counter()-t:.3f}s; KG={kg_verdict}; Topoly={tp_verdict}",
    )

    if strict and kg_verdict != "PASS":
        raise AssertionError(
            f"KnottedGraph vs LLLV theorem for {case.label}: {kg_verdict}"
        )

    row = {
        "family": case.family,
        "size": case.size,
        "label": case.label,
        "crossings": len(processor.crossings),
        "extra_projection_crossings": len(processor.crossings) - expected_crossings,
        "V": graph.number_of_nodes(),
        "E": graph.number_of_edges(),
        "max_degree": max(dict(graph.degree()).values()),
        "paper": LLLV_PAPER,
        "theorem": LLLV_THEOREM,
        "formula": LLLV_FORMULAS[case.family],
        "replacement": REPLACEMENT_NAME,
        "embedding_variant": EMBEDDING_VARIANT,
        "embedding_source": LLLV_PAPER,
        "reference_figure": LLLV_REFERENCE_FIGURES[case.family],
        "embedding_hash": projection["embedding_hash"],
        "projection_source": projection["projection_source"],
        "projection_s": projection["projection_s"],
        "projection_cache_key": projection["projection_cache_key"],
        "pd_hash": hashlib.sha256(pdcode.encode()).hexdigest(),
        "case_hash": hashlib.sha256(
            f"{EMBEDDING_VARIANT}:{case.family}:{case.size}:{REPLACEMENT_NAME}".encode()
        ).hexdigest()[:16],
        "knottedgraph_timeout_s": float(kg_timeout_s),
        "topoly_timeout_s": float(topoly_timeout_s),
        "knottedgraph_status": kg.get("status"),
        "knottedgraph_s": kg.get("time_s"),
        "knottedgraph_vs_theorem": kg_verdict,
    }
    if tp is not None:
        row.update(
            {
                "topoly_status": tp.get("status"),
                "topoly_s": tp.get("time_s"),
                "topoly_vs_theorem": tp_verdict,
            }
        )
    if kg.get("status") != "ok":
        row["knottedgraph_error"] = (
            kg.get("error")
            or f"KnottedGraph status: {kg.get('status')}; timeout={kg_timeout_s}s"
        )
    if tp is not None and tp.get("status") != "ok":
        row["topoly_error"] = (
            tp.get("error")
            or f"Topoly status: {tp.get('status')}; timeout={topoly_timeout_s}s"
        )
    if tp is not None and kg.get("time_s") and tp.get("time_s"):
        row["topoly_over_knottedgraph"] = tp["time_s"] / kg["time_s"]

    _stage_log(
        case_key,
        "CASE COMPUTE END",
        f"{time.perf_counter()-case_start:.3f}s before CSV merge/save",
    )
    return row



## 4. Run validation and timing

This is the main benchmark cell. It appends or replaces rows in the resume ledger, recomputes the KnottedGraph result, and records the Topoly result when the configured controller allows it.


In [ ]:
compare_dv_theta = compare_dv_theta_reference

assert LLLV_REPLACEMENT == REPLACEMENT_NAME
print("Reference embedding variant:", EMBEDDING_VARIANT)

all_result_rows = []

if RUN_DOBRYNIN_VESNIN:
    dv_settings_hash = _settings_hash(
        {
            "schema_version": RESULT_SCHEMA_VERSION,
            "benchmark": "dobrynin_vesnin_theta_validation",
            "embedding_variant": EMBEDDING_VARIANT,
            "projection_cache_schema": PROJECTION_CACHE_SCHEMA,
            "lllv_infinity_samples": LLLV_INFINITY_SAMPLES,
            "knottedgraph_timeout_s": THETA_KG_TIMEOUT_S,
            "topoly_timeout_s": THETA_TOPOLY_TIMEOUT_S,
        }
    )
    dv_rows = _run_cases_to_csv(
        list(THETA_N_VALUES),
        settings_hash=dv_settings_hash,
        run_case=lambda n, run_topoly: compare_dv_theta(
            n, THETA_KG_TIMEOUT_S, THETA_TOPOLY_TIMEOUT_S, strict=False, run_topoly=run_topoly
        ),
        case_key=lambda n: f"dv:theta:{int(n)}",
        case_family=lambda n: "dv_theta",
        case_size=lambda n: int(n),
        normalise_row=lambda raw: _normalise_dv_row(raw, settings_hash=dv_settings_hash, timeout_s=THETA_KG_TIMEOUT_S),
        error_row=lambda n, elapsed, exc: _error_row(
            settings_hash=dv_settings_hash,
            benchmark="dobrynin_vesnin_theta_validation",
            suite="Dobrynin-Vesnin Theta(n)",
            case_key=f"dv:theta:{int(n)}",
            graph_type=("dv_handcuff" if int(n) % 2 == 0 else "dv_theta"),
            size=int(n),
            label=f"Theta({int(n)})",
            timeout_s=THETA_KG_TIMEOUT_S,
            elapsed_s=elapsed,
            exc=exc,
            embedding_variant=EMBEDDING_VARIANT,
        ),
        desc="Dobrynin-Vesnin Theta(n): paper-reference embedding",
    )
    all_result_rows.extend(dv_rows)
else:
    dv_settings_hash = None
    dv_rows = []

if RUN_LI_LEI_LI_VESNIN:
    lllv_cases = build_lllv_reference_cases(
        cycle_n_values=LLLV_CYCLE_N_VALUES,
        theta_s_values=LLLV_THETA_S_VALUES,
        bouquet_q_values=LLLV_BOUQUET_Q_VALUES,
    )
    lllv_settings_hash = _settings_hash(
        {
            "schema_version": RESULT_SCHEMA_VERSION,
            "benchmark": "li_lei_li_vesnin_edge_replacement_validation",
            "embedding_variant": EMBEDDING_VARIANT,
            "projection_cache_schema": PROJECTION_CACHE_SCHEMA,
            "lllv_infinity_samples": LLLV_INFINITY_SAMPLES,
            "replacement": REPLACEMENT_NAME,
            "knottedgraph_timeout_s": LLLV_KG_TIMEOUT_S,
            "topoly_timeout_s": LLLV_TOPOLY_TIMEOUT_S,
        }
    )
    lllv_rows = _run_cases_to_csv(
        lllv_cases,
        settings_hash=lllv_settings_hash,
        run_case=lambda case, run_topoly: compare_lllv_reference_case(
            case, LLLV_KG_TIMEOUT_S, LLLV_TOPOLY_TIMEOUT_S, strict=False, run_topoly=run_topoly
        ),
        case_key=lambda case: case.case_key,
        case_family=lambda case: case.family,
        case_size=lambda case: case.size,
        normalise_row=lambda raw: _normalise_lllv_row(raw, settings_hash=lllv_settings_hash, timeout_s=LLLV_KG_TIMEOUT_S),
        error_row=lambda case, elapsed, exc: _error_row(
            settings_hash=lllv_settings_hash,
            benchmark="li_lei_li_vesnin_edge_replacement_validation",
            suite="Li-Lei-Li-Vesnin infinity_+ edge replacement",
            case_key=case.case_key,
            graph_type=case.family,
            size=case.size,
            label=case.label,
            timeout_s=LLLV_KG_TIMEOUT_S,
            elapsed_s=elapsed,
            exc=exc,
            embedding_variant=EMBEDDING_VARIANT,
            replacement=REPLACEMENT_NAME,
        ),
        desc="Li-Lei-Li-Vesnin infinity_+ paper embeddings",
    )
    all_result_rows.extend(lllv_rows)
else:
    lllv_settings_hash = None
    lllv_rows = []

print(f"Saved/reused {len(all_result_rows)} current rows")
print("CSV:", RESULTS_CSV)

summary_cols = [
    "suite", "graph_type", "size", "label", "crossings", "max_degree",
    "knottedgraph_s", "knottedgraph_result", "topoly_s", "topoly_result",
    "projection_source", "projection_s", "case_wall_s",
    "knottedgraph_timeout_s", "topoly_timeout_s",
]
print("\t".join(summary_cols))
for row in all_result_rows:
    print("\t".join(str(row.get(col, "")) for col in summary_cols))


> **Note on Topoly for larger diagrams.** Topoly documents `max_cross=15` as the default computational cutoff for polynomial evaluation; this is a runtime safeguard rather than a stated mathematical validity limit. In the version tested here (Topoly 1.1.0), supplying a larger value through `YamadaGraph.point(max_cross=...)` does not propagate that value to the recursive Yamada skein calls: the recursive calls invoke `.point(...)` without forwarding `max_cross` and therefore revert to the default setting. 

For the Dobrynin–Vesnin family considered in this benchmark, Topoly agrees with the published reference through $n=16$, returns a polynomial at $n=17$ that is not equivalent to the published reference polynomial, and fails to complete the  $n=18$ and $n=19$ evaluations. Direct inspection of the $n=18$ failure shows that recursive branches return Topoly's internal `'ErrTMC'` sentinel associated with its crossing-limit handling; this string is subsequently passed into Topoly's polynomial addition routine, which accepts only polynomial, term, integer, or floating-point operands and consequently raises `ValueError: Adding unsupported type.` Thus, the $n\ge18$ failures reported here should not be interpreted as evidence that Topoly is mathematically restricted to 15 crossings, but as a limitation of the crossing-limit propagation and error handling in Topoly 1.1.0 for these tested diagrams. KnottedGraph successfully reproduces the published polynomials for all cases.

## 6. Plot the scaling results

The plot reads the current in-memory rows when available, otherwise it reloads the CSV. Missing optional plotting dependencies only skip the figure; they do not change the recorded benchmark rows.


In [ ]:
%matplotlib notebook
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("matplotlib is not installed; skipping plots.")
else:
    plot_rows = list(all_result_rows) if "all_result_rows" in globals() else _read_result_csv()
    graph_order = ["dv_handcuff", "dv_theta", "cycle", "theta", "bouquet"]
    present = [graph for graph in graph_order if any(row.get("graph_type") == graph for row in plot_rows)]
    present += sorted({row.get("graph_type") for row in plot_rows if row.get("graph_type") and row.get("graph_type") not in present})

    if not present:
        print("No rows available to plot. Run the benchmark cell first.")
    else:
        fig, axes = plt.subplots(1, len(present), figsize=(5.0 * len(present), 4.6), sharey=True)
        if len(present) == 1:
            axes = [axes]

        framework_specs = [
            ("knottedgraph", "KnottedGraph", "tab:blue"),
            ("topoly", "Topoly", "tab:orange"),
        ]
        for ax, graph_type in zip(axes, present):
            rows = sorted(
                [row for row in plot_rows if row.get("graph_type") == graph_type],
                key=lambda row: (_as_float(row.get("size")) is None, _as_float(row.get("size")) or 0, str(row.get("label", ""))),
            )
            for framework, label, color in framework_specs:
                for passed, marker, suffix in [(True, ".", "pass"), (False, "x", "nonpass/error")]:
                    xs = []
                    ys = []
                    for row in rows:
                        result = row.get(f"{framework}_result") or row.get(f"{framework}_vs_published") or row.get(f"{framework}_vs_theorem")
                        y = _as_float(row.get(f"{framework}_s"))
                        x = _as_float(row.get("size"))
                        if x is None or y is None:
                            continue
                        is_pass = result == "PASS"
                        if is_pass == passed:
                            xs.append(x)
                            ys.append(y)
                    if xs:
                        ax.scatter(
                            xs,
                            ys,
                            marker=marker,
                            s=120 if marker == "." else 70,
                            color=color,
                            alpha=0.9,
                            label=f"{label} {suffix}",
                        )
            ax.set_title(graph_type)
            ax.set_xlabel("family parameter")
            ax.set_yscale("log")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8)
        axes[0].set_ylabel("runtime (s, log scale)")
        fig.suptitle("KnottedGraph vs Topoly runtime scaling; . = pass, x = nonpass/error")
        fig.tight_layout()
        plt.show()


## 7. Inspect pass/fail summaries

Use this summary after a long run to see which cases need attention. A KnottedGraph non-pass is a correctness issue for this benchmark; a Topoly non-pass records the tested Topoly behavior under the same reference input.


In [ ]:
current_rows = list(all_result_rows) if "all_result_rows" in globals() else _read_result_csv()
kg_nonpasses = [row for row in current_rows if _kg_result(row) != "PASS"]
topoly_nonpasses = [row for row in current_rows if _topoly_result(row) != "PASS"]

print("CSV:", RESULTS_CSV)
print("Current rows:", len(current_rows))
print("KnottedGraph nonpasses:", len(kg_nonpasses))
print("Topoly nonpasses:", len(topoly_nonpasses))

for row in kg_nonpasses:
    print("KG", row.get("case_key"), _kg_result(row), row.get("knottedgraph_error") or row.get("row_error", ""))
for row in topoly_nonpasses:
    print("Topoly", row.get("case_key"), _topoly_result(row), row.get("topoly_error") or row.get("row_error", ""))
